# MCTS

In [2]:
import numpy as np
from typing import Any
from dataclasses import dataclass

## Evaluador Dummy

In [3]:
class DummyEvaluator:
    """
    Evaluador dummy para pruebas.

    Devuelve:
    - priors/logits aleatorios para las 55 acciones del problema.
    - value aleatorio en [-1, 1].
    """

    def __init__(
        self,
        action_space_size: int = 55,
        random_seed: int | None = None,
    ):
        self.action_space_size = action_space_size
        self.rng = np.random.default_rng(random_seed)

    def predict(self, state: Any) -> tuple[np.ndarray, float]:
        priors = self.rng.normal(
            loc=0.0,
            scale=1.0,
            size=self.action_space_size,
        )

        value = float(self.rng.uniform(-1.0, 1.0))

        return priors, value

In [10]:
evaluator = DummyEvaluator()
fake_state = "df"
priors, value = evaluator.predict(fake_state)
print(priors)


[ 3.53772011e-01 -9.06877693e-01 -9.70252546e-01  2.55819478e-01
  4.90897702e-01  4.97503175e-01 -8.97859152e-01  1.26862610e+00
  1.21572733e+00  1.15250009e+00  1.28751619e+00  2.10250464e+00
  4.02294372e-01  6.18142259e-01 -9.19978545e-01 -2.79773841e-01
  8.18259792e-01  9.88045878e-02 -3.83136959e-01  4.79511310e-01
 -1.38406138e+00  1.03275587e+00  5.29120123e-01 -1.21511014e-03
  1.25463196e-01  9.73221550e-01  9.52531904e-01 -7.90923713e-01
 -2.50763208e-01 -1.47867733e+00 -1.01774134e-01  4.29954095e-01
  5.76648030e-01  1.90327994e+00 -5.75895432e-01 -3.19977896e-01
  3.17425013e-01  3.98830981e-01 -7.41412951e-01 -7.71553500e-02
  4.83441374e-01  5.17683479e-01 -4.66040336e-01 -1.00369202e-01
 -9.16946452e-01  5.13813889e-01  1.68160421e-01  2.46535854e-01
  1.30405083e+00  2.36888015e+00  1.41920136e-01 -3.79336934e-01
 -6.48234129e-01  2.37283545e-01 -9.39381924e-01]


In [ ]:
from typing import Any


def generate_trajectory(
    initial_state: Any,
    engine: Any,
    mcts: Any,
    debug: bool = False,
) -> tuple[list[dict[str, Any]], float, Any]:

    trajectory: list[dict[str, Any]] = []

    current_state = initial_state
    is_terminal = False
    final_reward = 0.0
    step_count = 0

    while not is_terminal:
        mcts_result = mcts.search(current_state)

        selected_action_id = mcts_result.selected_action_id
        policy = mcts_result.policy

        trajectory.append(
            {
                "state": current_state,
                "policy": policy,
                "action_id": selected_action_id,
                "reward": None,
            }
        )

        next_state, is_terminal, reward = engine.step(
            current_state,
            selected_action_id,
        )

        mcts.advance_root(selected_action_id)

        current_state = next_state
        step_count += 1

        if debug:
            print(
                f"[trajectory] step={step_count} | "
                f"action={selected_action_id} | "
                f"terminal={is_terminal} | "
                f"reward={reward}"
            )

        if is_terminal:
            final_reward = float(reward)

    for sample in trajectory:
        sample["reward"] = final_reward

    return trajectory, final_reward, current_state